In [1]:
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

## Extracting Nifty 200 stocks symbols

In [2]:
# Download the current NIFTY 200 constituent list
url = "https://nsearchives.nseindia.com/content/indices/ind_nifty200list.csv"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
response.raise_for_status()

stocks = pd.read_csv(StringIO(response.text))

# Convert NSE symbols to Yahoo Finance tickers
tickers = [symbol + ".NS" for symbol in stocks["Symbol"]]

print("Total stocks:", len(tickers))
print(tickers[:10])


Total stocks: 200
['360ONE.NS', 'ABB.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'ADANIENSOL.NS', 'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ATGL.NS']


## Downloading last 10 years of stock data

In [3]:
data = yf.download(
    tickers,
    period="10y",
    interval="1d",
    auto_adjust=False,
    group_by="ticker",
    threads=True,
    progress=True
)

print("Data shape:", data.shape)
display(data.head())

[*********************100%***********************]  200 of 200 completed


Data shape: (2472, 1200)


Ticker     HYUNDAI.NS                                 POLICYBZR.NS           \
Price            Open High Low Close Adj Close Volume         Open High Low   
Date                                                                          
2016-10-24        NaN  NaN NaN   NaN       NaN    NaN          NaN  NaN NaN   
2016-10-25        NaN  NaN NaN   NaN       NaN    NaN          NaN  NaN NaN   
2016-10-26        NaN  NaN NaN   NaN       NaN    NaN          NaN  NaN NaN   
2016-10-27        NaN  NaN NaN   NaN       NaN    NaN          NaN  NaN NaN   
2016-10-28        NaN  NaN NaN   NaN       NaN    NaN          NaN  NaN NaN   

Ticker            ...   BIOCON.NS                                    \
Price      Close  ...         Low       Close   Adj Close    Volume   
Date              ...                                                 
2016-10-24   NaN  ...  161.116669  161.774994  158.993530   9018696   
2016-10-25   NaN  ...  156.250000  156.583328  153.891113   9939390   
2016-10-26   NaN  ...  149.725006  151.183334  148.583969   9286560   
2016-10-27   NaN  ...  149.149994  153.475006  150.836258  14619996   
2016-10-28   NaN  ...  153.366669  154.274994  151.622467   5233548   

Ticker     RELIANCE.NS                                                  \
Price             Open        High         Low       Close   Adj Close   
Date                                                                     
2016-10-24  244.584000  245.498337  242.446747  243.052490  231.921814   
2016-10-25  243.441086  243.498230  240.972382  242.492462  231.387436   
2016-10-26  241.395264  242.172455  240.492355  240.938095  229.904236   
2016-10-27  240.263779  240.903809  238.503693  239.715179  228.737335   
2016-10-28  240.012344  241.841003  238.595123  241.143829  230.100555   

Ticker                
Price         Volume  
Date                  
2016-10-24  10513017  
2016-10-25   7658966  
2016-10-26  10945927  
2016-10-27  12842476  
2016-10-28   8127969  

[5 rows x 1200 columns]

## Extracting Adjusted Closing Price 
We use adjusted closing prices because they account for corporate actions
such as stock splits and dividends, providing a more consistent historical
price series and reducing artificial distortions in return calculations
and statistical relationships between stocks.

In [4]:
adj_close = data.xs("Adj Close",axis = 1,level = 1)
print("Dataset size",adj_close.shape)

Dataset size (2472, 200)


In [5]:
adj_close.head()

Ticker,HYUNDAI.NS,POLICYBZR.NS,GMRAIRPORT.NS,ZYDUSLIFE.NS,UNIONBANK.NS,PAGEIND.NS,ADANIENT.NS,HEROMOTOCO.NS,BAJAJFINSV.NS,RVNL.NS,...,TECHM.NS,SRF.NS,ADANIENSOL.NS,ONGC.NS,M&MFIN.NS,BSE.NS,APLAPOLLO.NS,GODREJPROP.NS,BIOCON.NS,RELIANCE.NS
Date,,,,,,,,,,,,,,,,,,,,,
2016-10-24,NaN,NaN,12.072441,358.979675,124.154854,15119.436523,38.094032,2406.681396,320.584747,NaN,...,306.704529,360.606873,44.049999,115.603836,189.548035,NaN,90.029739,363.891052,158.993530,231.921814
2016-10-25,NaN,NaN,11.892256,376.084930,123.945145,15004.694336,36.013271,2421.620605,323.336761,NaN,...,310.206512,359.132263,43.450001,114.180313,188.990692,NaN,88.226181,362.796326,153.891113,231.387436
2016-10-26,NaN,NaN,12.477859,357.965393,121.344597,15067.112305,35.906563,2467.447266,331.234619,NaN,...,302.345001,353.875092,43.200001,114.714111,192.626633,NaN,88.838539,360.855743,148.583969,229.904236
2016-10-27,NaN,NaN,12.252627,364.327942,120.673515,14698.842773,35.719822,2391.382324,325.272675,NaN,...,295.948730,346.597443,43.200001,114.990913,193.714783,NaN,87.350708,356.377380,150.836258,228.737335
2016-10-28,NaN,NaN,12.342720,390.008881,123.861244,14765.655273,35.693150,2418.949951,336.733734,NaN,...,311.349976,348.742401,45.250000,113.448730,192.095871,NaN,87.719093,357.322784,151.622467,230.100555


## Removing the stocks which are recently listed or are not in the Nifty 200 portfolio from the beginning 


In [6]:
# Find the first available trading date for each stock
first_dates = adj_close.apply(pd.Series.first_valid_index)

# Keep stocks that have data from the beginning of our dataset
start_date = adj_close.index.min()

valid_stocks = first_dates[first_dates <= start_date].index

# Filter the price matrix
df = adj_close[valid_stocks].copy()

# Check results
print("Original stocks:", adj_close.shape[1])
print("Stocks retained:", df.shape[1])
print("Stocks removed:", adj_close.shape[1] - df.shape[1])

display(df.head())

Original stocks: 200
Stocks retained: 150
Stocks removed: 50


Ticker,GMRAIRPORT.NS,ZYDUSLIFE.NS,UNIONBANK.NS,PAGEIND.NS,ADANIENT.NS,HEROMOTOCO.NS,BAJAJFINSV.NS,KEI.NS,MRF.NS,HINDZINC.NS,...,TRENT.NS,TECHM.NS,SRF.NS,ADANIENSOL.NS,ONGC.NS,M&MFIN.NS,APLAPOLLO.NS,GODREJPROP.NS,BIOCON.NS,RELIANCE.NS
Date,,,,,,,,,,,,,,,,,,,,,
2016-10-24,12.072441,358.979675,124.154854,15119.436523,38.094032,2406.681396,320.584747,117.062599,49220.449219,113.242607,...,200.490036,306.704529,360.606873,44.049999,115.603836,189.548035,90.029739,363.891052,158.993530,231.921814
2016-10-25,11.892256,376.084930,123.945145,15004.694336,36.013271,2421.620605,323.336761,116.232712,49368.460938,113.868614,...,201.230392,310.206512,359.132263,43.450001,114.180313,188.990692,88.226181,362.796326,153.891113,231.387436
2016-10-26,12.477859,357.965393,121.344597,15067.112305,35.906563,2467.447266,331.234619,113.938316,48913.683594,113.220245,...,196.541443,302.345001,353.875092,43.200001,114.714111,192.626633,88.838539,360.855743,148.583969,229.904236
2016-10-27,12.252627,364.327942,120.673515,14698.842773,35.719822,2391.382324,325.272675,113.694237,45725.972656,112.147079,...,195.258163,295.948730,346.597443,43.200001,114.990913,193.714783,87.350708,356.377380,150.836258,228.737335
2016-10-28,12.342720,390.008881,123.861244,14765.655273,35.693150,2418.949951,336.733734,114.621750,47255.269531,113.108452,...,196.837570,311.349976,348.742401,45.250000,113.448730,192.095871,87.719093,357.322784,151.622467,230.100555


In [13]:
df.isnull().sum()[df.isnull().sum() > 0]

Ticker
PAGEIND.NS       1
MRF.NS           1
RADICO.NS        1
PHOENIXLTD.NS    1
SOLARINDS.NS     1
BHARATFORG.NS    1
AUROPHARMA.NS    1
SIEMENS.NS       1
dtype: int64

In [16]:
df=df.dropna(axis=0)
print(df.isnull().sum()[df.isnull().sum()>0])

Series([], dtype: int64)


In [17]:
# Save the cleaned NIFTY 200 adjusted closing price dataset
df.to_csv("nifty200_cleaned.csv", index=True)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
